In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

import matplotlib as mpl
from multiprocess import Pool
from scipy import optimize
from scipy.stats import norm

from AnalysePoisson import *  
from AnalyseNegBinomial import *  

# Define chip simulation

In [ ]:
def CreateAndAnalyseChip_NegBinom(lambda_NegBinom,r,q,rho,N):
    
    chip=CreateNegBinomChip(lambda_NegBinom,r,q,rho,N)
    return AnalyseNegBinomChip(chip)    

def CreateNegBinomChip(lambda_NegBinom,r,q,rho,N):
    chip = pd.DataFrame(np.random.negative_binomial(r,r/(lambda_NegBinom+r), N),columns=['cells'])
    chip['dead']=chip['cells'].apply(lambda n: np.random.choice([1,0], size=None, replace=True, p=[np.power(q,n),1-np.power(q,n)]))
    chip['detected_cells']=chip['cells'].apply(lambda n: np.random.binomial(n, rho, 1)[0])
    
    return chip


def CreateAndAnalyseChip_Poiss(lambda_Poiss,q,rho,N):
    
    chip=CreatePoissonianChip(lambda_Poiss,q,rho,N)
    return AnalysePoissonianChip(chip)    

def CreatePoissonianChip(lambda_Poiss,q,rho,N):
    chip = pd.DataFrame(np.random.poisson(lambda_Poiss, N),columns=['cells'])
    chip['dead']=chip['cells'].apply(lambda n: np.random.choice([1,0], size=None, replace=True, p=[np.power(q,n),1-np.power(q,n)]))
    chip['detected_cells']=chip['cells'].apply(lambda n: np.random.binomial(n, rho, 1)[0])
    
    return chip



# Define Parameters and sample many chips: Poisson

In [ ]:
colors=sns.color_palette("colorblind")


In [ ]:
N_droplets_per_chip=500
Num_chips=100
detection_prob=1

lambda_Poiss=3
q=0.9

In [ ]:
chip = pd.concat(map(lambda x: CreateAndAnalyseChip_Poiss(lambda_Poiss,q,detection_prob,N_droplets_per_chip), np.arange(Num_chips)),ignore_index=True)

## calculate Fisher Information

In [ ]:
num_samples=10000 # for Monte_Carlo estimate
Fisher_Information=Fisher_Information_Poiss([lambda_Poiss * (1-detection_prob),q],lambda_Poiss * detection_prob,num_samples) 
Cov=ComputeCov_Poiss(Fisher_Information,N_droplets_per_chip,lambda_Poiss * detection_prob,detection_prob,lambda_Poiss,q) # includes sample size 

In [ ]:
Estimate_Fisher_Information_Poiss([1,q],lambda_Poiss * detection_prob,num_samples)

## Plots

### Parameters of Initial distribution $ \tilde{\lambda} = \lambda \rho$

In [ ]:
height=8
lw=8

fig = plt.figure(figsize=(20, 10), dpi=200)

sns.kdeplot(data=chip['lambda_rho'],c=colors[0],linewidth=lw,)
plt.plot(lambda_Poiss * detection_prob * np.ones(height),range(height),'--', c='gray',linewidth=lw) #target value

x = np.linspace(0, 3, 100)
plt.plot(x, norm.pdf(x, lambda_Poiss * detection_prob, np.sqrt(lambda_Poiss * detection_prob/N_droplets_per_chip)),c='gray',linewidth=lw) # sampling distribution 

plt.xlabel(r'$\tilde{\lambda}$')
plt.ylabel('pdf')
plt.xlim([2,3])

plt.rcParams.update({'font.size': 30})
plt.legend(frameon=False)

### $\Lambda = \lambda (1-\rho)$,  $\lambda$, $\rho$

In [ ]:
height=4
lw=8

fig = plt.figure(figsize=(20, 10), dpi=200)
sns.kdeplot(data=chip['Lam_like'],c=colors[0],linewidth=lw,clip=(0,np.inf))
plt.plot(lambda_Poiss * (1-detection_prob) * np.ones(height),range(height),'--', c='gray',linewidth=lw) #target value

x = np.linspace(0, 3, 100)
plt.plot(x, norm.pdf(x, lambda_Poiss * (1-detection_prob), np.sqrt(1/(N_droplets_per_chip * Fisher_Information[0] ))),c='gray',linewidth=lw) # Gaussian asymptotic sampling distribution


plt.xlabel(r'$\Lambda$')
plt.ylabel('pdf')
plt.xlim([0,3])

plt.rcParams.update({'font.size': 30})
plt.legend(frameon=False)

In [ ]:
height=11
lw=8

fig = plt.figure(figsize=(20, 10), dpi=200)
sns.kdeplot(data=chip['rho_like'],c=colors[0],clip=(0,1),linewidth=lw,label='MLE') # from maximum liklihood estimate
sns.kdeplot(data=chip['rho'],c=colors[1],clip=(0,1),linewidth=lw,label=r'$1- \frac{\log(\hat{d}_-)}{\log(\hat{q}_{|\tilde{0}})}$')
plt.plot(detection_prob * np.ones(height),range(height),'--', c='gray',linewidth=lw) #target value

x = np.linspace(0, 2, 1000)
plt.plot(x, norm.pdf(x, detection_prob, np.sqrt(Cov[1,1])),c='gray',linewidth=lw) # Gaussian asymptotic sampling distribution
plt.xlim([0.5,1])

plt.xlabel(r'$\rho$')
plt.ylabel('pdf')
plt.rcParams.update({'font.size': 30})
plt.legend(frameon=False)

In [ ]:
height=4
lw=8
fig = plt.figure(figsize=(20, 10), dpi=200)
sns.kdeplot(data=chip['lambda_like'],c=colors[0],clip=(0,np.inf),linewidth=lw,label='MLE') # from maximum liklihood estimate
sns.kdeplot(data=chip['lambda'],c=colors[1],clip=(0,np.inf),linewidth=lw,label=r'$\frac{\hat{\tilde{\lambda}}}{1- \frac{\log(\hat{d}_-)}{\log(\hat{q}_{|\tilde{0}})}}$')
plt.plot(lambda_Poiss * np.ones(height),range(height),'--', c='gray',linewidth=lw) #target value

x = np.linspace(0, 5, 1000)
plt.plot(x, norm.pdf(x, lambda_Poiss, np.sqrt(Cov[0,0])),c='gray',linewidth=lw) # Gaussian asymptotic sampling distribution



plt.xlabel(r'$\lambda$')
plt.ylabel('pdf')
plt.xlim([2,3.5])
plt.rcParams.update({'font.size': 30})
plt.legend(frameon=False)

### q

In [ ]:
height=11
lw=8
fig = plt.figure(figsize=(20, 10), dpi=200)
sns.kdeplot(data=chip['q_chip_like'],c=colors[0],clip=(0,np.inf),linewidth=lw,label='MLE')
sns.kdeplot(data=chip['q_chip'],c=colors[1],clip=(0,np.inf),linewidth=lw,label=r'$1-\frac{\log \frac{\hat{q}_{|\tilde{0}}}{\hat{d}_-}}{\hat{\tilde{\lambda}}}$')
plt.plot(q * np.ones(height),range(height),'--', c='gray',linewidth=lw) # target value

x = np.linspace(0, 1, 1000)
plt.plot(x, norm.pdf(x, q, np.sqrt(1/(N_droplets_per_chip * Fisher_Information[2] ))),c='gray',linewidth=lw) # Gaussian asymptotic sampling distribution
plt.plot(x, norm.pdf(x,q, np.sqrt(Cov[2,2])),c='gray',linewidth=lw) # Gaussian asymptotic sampling distribution

plt.xlim([0.5,1])

plt.xlabel(r'$q$')
plt.ylabel('pdf')
plt.rcParams.update({'font.size': 30})
plt.legend(frameon=False)

# Define Parameters and sample many chips: Negative Binomial

In [ ]:
N_droplets_per_chip=500
Num_chips=1000
detection_prob=0.8
lambda_NegBinom=3
r=2
q=0.8

In [ ]:
chip = pd.concat(map(lambda x: CreateAndAnalyseChip_NegBinom(lambda_NegBinom,r,q,detection_prob,N_droplets_per_chip), np.arange(Num_chips)),ignore_index=True)

## Calculate FI

In [ ]:
num_samples=10000 # for Monte_Carlo estimate
Fisher_Information=   Fisher_Information_NegBinom([(lambda_NegBinom * (1-detection_prob))/(r+lambda_NegBinom * detection_prob),q],lambda_NegBinom * detection_prob,r,num_samples)
    
FisherInfo_r=  N_droplets_per_chip * Estimate_Fisher_Information_part_r(lambda_NegBinom * detection_prob,r,num_samples) 

b=q/(1+(lambda_NegBinom * (1-detection_prob))/(r+lambda_NegBinom * detection_prob) * (1-q))
a= np.power(b/q,r)

Cov=ComputeCov_NegBinom(Fisher_Information,FisherInfo_r,N_droplets_per_chip,lambda_NegBinom * detection_prob,detection_prob,lambda_NegBinom,r,(lambda_NegBinom * (1-detection_prob))/(r+lambda_NegBinom * detection_prob),q)


phi=(lambda_NegBinom * (1-detection_prob))/(r+lambda_NegBinom * detection_prob)
A=(1-detection_prob)/lambda_NegBinom
B=-detection_prob/lambda_NegBinom

## Plots

### Initial distribution

In [ ]:
height=8
lw=8

fig = plt.figure(figsize=(20, 10), dpi=200)

sns.kdeplot(data=chip['lambda_rho'],c=colors[0],linewidth=lw)
plt.plot(lambda_NegBinom * detection_prob * np.ones(height),range(height),'--', c='gray',linewidth=lw)

x = np.linspace(0, 3, 100)
plt.plot(x, norm.pdf(x, lambda_NegBinom * detection_prob, np.sqrt(lambda_NegBinom * detection_prob/N_droplets_per_chip)),c='gray',linewidth=lw)

plt.xlabel(r'$\lambda \rho$')
plt.ylabel('pdf')
plt.xlim([2,3])

plt.rcParams.update({'font.size': 30})
plt.legend(frameon=False)

In [ ]:
height=8
lw=8

fig = plt.figure(figsize=(20, 10), dpi=200)


sns.kdeplot(data=chip['r'],c=colors[0],clip=(0,np.inf),linewidth=lw)
sns.kdeplot(data=chip['r_like'],c=colors[1],clip=(0,np.inf),linewidth=lw)



plt.plot(r * np.ones(height),range(height),'--', c='gray',linewidth=lw)
x = np.linspace(0, 3, 100)

plt.plot(x, norm.pdf(x, r, np.sqrt(1/FisherInfo_r)),c='gray',linewidth=lw)
plt.plot(x, norm.pdf(x, chip['r_like'].mean(), np.sqrt(Cov[1,1])),c='r',linewidth=lw)

plt.xlabel(r'$r$')
plt.ylabel('pdf')
plt.xlim([0,3])

plt.rcParams.update({'font.size': 30})
plt.legend(frameon=False)



### $\Phi= \frac{\lambda (1-\rho)}{r + \lambda \rho}, \lambda, \rho$

In [ ]:
height=8
lw=8

fig = plt.figure(figsize=(20, 10), dpi=200)

sns.kdeplot(data=chip['Phi_like'],c=colors[0],linewidth=lw,)
plt.plot((lambda_NegBinom * (1-detection_prob))/(r+lambda_NegBinom * detection_prob)* np.ones(height),range(height),'--', c='gray',linewidth=lw) # target

x = np.linspace(0, 3, 100)
plt.plot(x, norm.pdf(x, (lambda_NegBinom * (1-detection_prob))/(r+lambda_NegBinom * detection_prob), np.sqrt(1/(N_droplets_per_chip * Fisher_Information[0] ))),c='gray',linewidth=lw)

plt.xlabel(r'$\Phi$')
plt.ylabel('pdf')
plt.xlim([0,1])

plt.rcParams.update({'font.size': 30})
plt.legend(frameon=False)

In [ ]:
height=11
lw=8

fig = plt.figure(figsize=(20, 10), dpi=200)
sns.kdeplot(data=chip['rho_like'],c=colors[0],clip=(0,1),linewidth=lw,label='MLE')
sns.kdeplot(data=chip['rho'],c=colors[1],clip=(0,1),linewidth=lw,label='')
plt.plot(detection_prob * np.ones(height),range(height),'--', c='gray',linewidth=lw)


x = np.linspace(0, 2, 1000)
plt.plot(x, norm.pdf(x, detection_prob, np.sqrt(Cov[2,2])),c='gray',linewidth=lw)
plt.xlim([0,1])

plt.xlabel(r'$\rho$')
plt.ylabel('pdf')
plt.rcParams.update({'font.size': 30})
plt.legend(frameon=False)

In [ ]:
height=4
lw=8

fig = plt.figure(figsize=(20, 10), dpi=200)
sns.kdeplot(data=chip['lambda_like'],c=colors[0],clip=(0,np.inf),linewidth=lw,label='MLE')
sns.kdeplot(data=chip['lambda'],c=colors[1],clip=(0,np.inf),linewidth=lw,label='')
plt.plot(lambda_NegBinom * np.ones(height),range(height),'--', c='gray',linewidth=lw)

x = np.linspace(0, 5, 1000)
plt.plot(x, norm.pdf(x, lambda_NegBinom, np.sqrt(Cov[0,0])),c='gray',linewidth=lw)


plt.xlabel(r'$\lambda$')
plt.ylabel('pdf')
plt.xlim([2,3.5])
plt.rcParams.update({'font.size': 30})
plt.legend(frameon=False)

### q

In [ ]:
height=31
lw=8

fig = plt.figure(figsize=(20, 10), dpi=200)
sns.kdeplot(data=chip['q_chip_like'],c=colors[0],clip=(0,1),linewidth=lw,label='MLE')
sns.kdeplot(data=chip['q_chip'],c=colors[1],clip=(0,1),linewidth=lw,label=r'$\frac{1 + \frac{\hat{r}}{\hat{\tilde{\lambda}}} \left(1-\sqrt[\hat{r}]{\frac{\hat{q}_{|\tilde{0}}}{\hat{d}_-}}\right)}{\sqrt[\hat{r}]{\hat{q}_{|\tilde{0}}}}$')  
plt.plot(q * np.ones(height),range(height),'--', c='gray',linewidth=lw) # target


x = np.linspace(0, 1, 1000)
plt.plot(x, norm.pdf(x, q, np.sqrt(1/(N_droplets_per_chip * Fisher_Information[2] ))),c='gray',linewidth=lw) # Gaussian sampling distribution
plt.plot(x, norm.pdf(x, chip['q_chip_like'].mean(), np.sqrt(Cov[3,3])),c='r',linewidth=lw)

plt.xlim([0.5,1])

plt.xlabel(r'$q$')
plt.ylabel('pdf')
plt.rcParams.update({'font.size': 30})
plt.legend(frameon=False)

In [ ]:
height=25
lw=8

colors=sns.color_palette("colorblind")

fig = plt.figure(figsize=(20, 10), dpi=200)




sns.kdeplot(data=chip['q_chip_like'],c=colors[0],clip=(0,1),linewidth=lw,label='MLE')
sns.kdeplot(data=chip['q_chip_uncor'],c=colors[1],clip=(0,1),linewidth=lw,label=r'$1 + \frac{\hat{r}}{\hat{\tilde{\lambda}}} \left(1-\sqrt[\hat{r}]{\frac{1}{\hat{d}_-}}\right)}{\sqrt[\hat{r}]$')
sns.kdeplot(data=chip['q_single'],c=colors[2],clip=(0,1),linewidth=lw,label=r'$\frac{\hat{q}_{\tilde{1}}}{\hat{q}_{|\tilde{0}} \ \sqrt[\hat{r}]{\hat{q}_{|\tilde{0}}}}$')
sns.kdeplot(data=chip['prob_neg_drop_one_det'],c=colors[3],clip=(0,1),linewidth=lw,label=r'$\hat{q}_{\tilde{1}}$')

plt.plot(q * np.ones(height),range(height),'--', c='gray',linewidth=lw) # target


x = np.linspace(0, 1, 10000)
plt.plot(x, norm.pdf(x, q, np.sqrt(Cov[3,3])), c='gray',linewidth=lw)

#plt.plot(x, norm.pdf(x, q, np.sqrt(1/(N * Fisher_Information[2] ))),c='k',linewidth=4)
#plt.plot(x, norm.pdf(x, q, np.sqrt(Cov[3,3])), c='gray',linewidth=4,label='theory')

plt.xlabel('q')
plt.ylabel('pdf')


plt.rcParams.update({'font.size': 40})
plt.legend(frameon=False)

plt.xlim([0,1])

# Poisson: Sample  3 different values of q

In [ ]:
N_droplets_per_chip=500
Num_chips=200
detection_prob=0.8
lambda_Poiss=3

q_1=0.1
q_2=0.5
q_3=0.9

In [ ]:
chip_1 = pd.concat(map(lambda x: CreateAndAnalyseChip_Poiss(lambda_Poiss,q_1,detection_prob,N_droplets_per_chip), np.arange(Num_chips)),ignore_index=True)
chip_2 = pd.concat(map(lambda x: CreateAndAnalyseChip_Poiss(lambda_Poiss,q_2,detection_prob,N_droplets_per_chip), np.arange(Num_chips)),ignore_index=True)
chip_3 = pd.concat(map(lambda x: CreateAndAnalyseChip_Poiss(lambda_Poiss,q_3,detection_prob,N_droplets_per_chip), np.arange(Num_chips)),ignore_index=True)

In [ ]:
num_samples=10000 # for Monte_Carlo estimate

Fisher_Information_1=Fisher_Information_Poiss([lambda_Poiss * (1-detection_prob),q_1],lambda_Poiss *  detection_prob,num_samples) 
Fisher_Information_2=Fisher_Information_Poiss([lambda_Poiss * (1-detection_prob),q_2],lambda_Poiss *  detection_prob,num_samples) 
Fisher_Information_3=Fisher_Information_Poiss([lambda_Poiss * (1-detection_prob),q_3],lambda_Poiss *  detection_prob,num_samples) 

Cov_1=ComputeCov_Poiss(Fisher_Information_1,N_droplets_per_chip,lambda_Poiss * detection_prob,detection_prob,lambda_Poiss,q_1) # includes sample size 
Cov_2=ComputeCov_Poiss(Fisher_Information_2,N_droplets_per_chip,lambda_Poiss * detection_prob,detection_prob,lambda_Poiss,q_2) # includes sample size 
Cov_3=ComputeCov_Poiss(Fisher_Information_3,N_droplets_per_chip,lambda_Poiss * detection_prob,detection_prob,lambda_Poiss,q_3) # includes sample size 

In [ ]:
height=60
lw=8
lw2=4

colors=sns.color_palette("colorblind")

fig = plt.figure(figsize=(20, 10), dpi=200)
x = np.linspace(0, 1, 10000)


plt.plot(q_1 * np.ones(height),range(height),'-', c='black',linewidth=lw2,linestyle='-') # target
plt.plot(x, norm.pdf(x, q_1, np.sqrt(Cov_1[2,2])), c='black',linewidth=lw,label='sampling',linestyle='-')

sns.kdeplot(data=chip_1['q_chip_like'],c=colors[0],clip=(0,1),linewidth=lw,label='MLE',linestyle='-')
sns.kdeplot(data=chip_1['q_chip'],c=colors[1],clip=(0,1),linewidth=lw,label=r'$1+\frac{\ln \left(\frac{\hat{d}_-}{\hat{q}_{|\tilde{0}}} \right)}{\hat{\tilde{\lambda}}}$',linestyle='-')  
sns.kdeplot(data=chip_1['q_chip_uncor'],c=colors[4],clip=(0,1),linewidth=lw,label=r'$1+\frac{\ln \left( \hat{d}_- \right)}{\hat{\tilde{\lambda}}}$',linestyle='-')
sns.kdeplot(data=chip_1['q_single'],c=colors[2],clip=(0,1),linewidth=lw,label=r'$\frac{\hat{q}_{\tilde{1}}}{\hat{q}_{|\tilde{0}}}$',linestyle='-')
sns.kdeplot(data=chip_1['prob_neg_drop_one_det'],c=colors[3],clip=(0,1),linewidth=lw,label=r'$\hat{q}_{\tilde{1}}$',linestyle='-')



plt.plot(q_2 * np.ones(height),range(height),'--', c='black',linewidth=lw2,linestyle='--') # target
plt.plot(x, norm.pdf(x, q_2, np.sqrt(Cov_2[2,2])), c='black',linewidth=lw,linestyle='--')
sns.kdeplot(data=chip_2['q_chip_like'],c=colors[0],clip=(0,1),linewidth=lw,linestyle='--')
sns.kdeplot(data=chip_2['q_chip'],c=colors[1],clip=(0,1),linewidth=lw,linestyle='--')
sns.kdeplot(data=chip_2['q_chip_uncor'],c=colors[4],clip=(0,1),linewidth=lw,linestyle='--')
sns.kdeplot(data=chip_2['q_single'],c=colors[2],clip=(0,1),linewidth=lw,linestyle='--')
sns.kdeplot(data=chip_2['prob_neg_drop_one_det'],c=colors[3],clip=(0,1),linewidth=lw,linestyle='--')


plt.plot(q_3 * np.ones(height),range(height),'--', c='black',linewidth=lw2,linestyle='-') # target
plt.plot(x, norm.pdf(x, q_3, np.sqrt(Cov_3[2,2])), c='black',linewidth=lw,linestyle='-')
sns.kdeplot(data=chip_3['q_chip_like'],c=colors[0],clip=(0,1),linewidth=lw,linestyle='-')
sns.kdeplot(data=chip_3['q_chip'],c=colors[1],clip=(0,1),linewidth=lw,linestyle='-')
sns.kdeplot(data=chip_3['q_chip_uncor'],c=colors[4],clip=(0,1),linewidth=lw,linestyle='-')
sns.kdeplot(data=chip_3['q_single'],c=colors[2],clip=(0,1),linewidth=lw,linestyle='-')
sns.kdeplot(data=chip_3['prob_neg_drop_one_det'],c=colors[3],clip=(0,1),linewidth=lw,linestyle='-')




plt.xlabel('q')
plt.ylabel('pdf')


plt.rcParams.update({'font.size': 30})
plt.legend(frameon=False,bbox_to_anchor=(0.15,0.2))

plt.xlim([-0.005,1.005])
plt.ylim([0,40])

# Neg. Binom: Sample  3 different values of q 

In [ ]:
N_droplets_per_chip=500
Num_chips=200
detection_prob=0.8
lambda_NegBinom=3
r=2
q_1=0.1
q_2=0.5
q_3=0.9

In [ ]:
chip_1 = pd.concat(map(lambda x: CreateAndAnalyseChip_NegBinom(lambda_NegBinom,r,q_1,detection_prob,N_droplets_per_chip), np.arange(Num_chips)),ignore_index=True)
chip_2 = pd.concat(map(lambda x: CreateAndAnalyseChip_NegBinom(lambda_NegBinom,r,q_2,detection_prob,N_droplets_per_chip), np.arange(Num_chips)),ignore_index=True)
chip_3 = pd.concat(map(lambda x: CreateAndAnalyseChip_NegBinom(lambda_NegBinom,r,q_3,detection_prob,N_droplets_per_chip), np.arange(Num_chips)),ignore_index=True)

In [ ]:
num_samples=10000 # for Monte_Carlo estimate
FisherInfo_r=  N_droplets_per_chip * Estimate_Fisher_Information_part_r(lambda_NegBinom * detection_prob,r,num_samples) 

Fisher_Information_1=   Fisher_Information_NegBinom([(lambda_NegBinom * (1-detection_prob))/(r+lambda_NegBinom * detection_prob),q_1],lambda_NegBinom * detection_prob,r,num_samples)
Fisher_Information_2=   Fisher_Information_NegBinom([(lambda_NegBinom * (1-detection_prob))/(r+lambda_NegBinom * detection_prob),q_2],lambda_NegBinom * detection_prob,r,num_samples)
Fisher_Information_3=   Fisher_Information_NegBinom([(lambda_NegBinom * (1-detection_prob))/(r+lambda_NegBinom * detection_prob),q_3],lambda_NegBinom * detection_prob,r,num_samples)
    

Cov_1=ComputeCov_NegBinom(Fisher_Information_1,FisherInfo_r,N_droplets_per_chip,lambda_NegBinom * detection_prob,detection_prob,lambda_NegBinom,r,(lambda_NegBinom * (1-detection_prob))/(r+lambda_NegBinom * detection_prob),q_1)
Cov_2=ComputeCov_NegBinom(Fisher_Information_2,FisherInfo_r,N_droplets_per_chip,lambda_NegBinom * detection_prob,detection_prob,lambda_NegBinom,r,(lambda_NegBinom * (1-detection_prob))/(r+lambda_NegBinom * detection_prob),q_2)
Cov_3=ComputeCov_NegBinom(Fisher_Information_3,FisherInfo_r,N_droplets_per_chip,lambda_NegBinom * detection_prob,detection_prob,lambda_NegBinom,r,(lambda_NegBinom * (1-detection_prob))/(r+lambda_NegBinom * detection_prob),q_3)

In [ ]:
height=35
lw=8
lw2=4

colors=sns.color_palette("colorblind")

fig = plt.figure(figsize=(20, 10), dpi=200)
x = np.linspace(0, 1, 10000)


plt.plot(q_1 * np.ones(height),range(height),'-', c='black',linewidth=lw2,linestyle='--') # target
plt.plot(x, norm.pdf(x, q_1, np.sqrt(Cov_1[3,3])), c='black',linewidth=lw,label='sampling',linestyle='--')
sns.kdeplot(data=chip_1['q_chip_like'],c=colors[0],clip=(0,1),linewidth=lw,label='MLE',linestyle='--')
sns.kdeplot(data=chip_1['q_chip'],c=colors[1],clip=(0,1),linewidth=lw,label=r'$\frac{1 + \frac{\hat{r}}{\hat{\tilde{\lambda}}} \left(1-\sqrt[\hat{r}]{\frac{\hat{q}_{|\tilde{0}}}{\hat{d}_-}}\right)}{\sqrt[\hat{r}]{\hat{q}_{|\tilde{0}}}}$',linestyle='--')  
sns.kdeplot(data=chip_1['q_chip_uncor'],c=colors[4],clip=(0,1),linewidth=lw,label=r'$1 + \frac{\hat{r}}{\hat{\tilde{\lambda}}} \left(1-\sqrt[\hat{r}]{\frac{1}{\hat{d}_-}}\right)}{\sqrt[\hat{r}]$',linestyle='--')
sns.kdeplot(data=chip_1['q_single'],c=colors[2],clip=(0,1),linewidth=lw,label=r'$\frac{\hat{q}_{\tilde{1}}}{\hat{q}_{|\tilde{0}} \ \sqrt[\hat{r}]{\hat{q}_{|\tilde{0}}}}$',linestyle='--')
sns.kdeplot(data=chip_1['prob_neg_drop_one_det'],c=colors[3],clip=(0,1),linewidth=lw,label=r'$\hat{q}_{\tilde{1}}$',linestyle='--')



plt.plot(q_2 * np.ones(height),range(height),'--', c='black',linewidth=lw2,linestyle='-') # target
plt.plot(x, norm.pdf(x, q_2, np.sqrt(Cov_2[3,3])), c='black',linewidth=lw,linestyle='-')
sns.kdeplot(data=chip_2['q_chip_like'],c=colors[0],clip=(0,1),linewidth=lw,linestyle='-')
sns.kdeplot(data=chip_2['q_chip'],c=colors[1],clip=(0,1),linewidth=lw,linestyle='-')
sns.kdeplot(data=chip_2['q_chip_uncor'],c=colors[4],clip=(0,1),linewidth=lw,linestyle='-')

sns.kdeplot(data=chip_2['q_single'],c=colors[2],clip=(0,1),linewidth=lw,linestyle='-')
sns.kdeplot(data=chip_2['prob_neg_drop_one_det'],c=colors[3],clip=(0,1),linewidth=lw,linestyle='-')


plt.plot(q_3 * np.ones(height),range(height),'--', c='black',linewidth=lw2,linestyle='-.') # target
plt.plot(x, norm.pdf(x, q_3, np.sqrt(Cov_3[3,3])), c='black',linewidth=lw,linestyle='-.')
sns.kdeplot(data=chip_3['q_chip_like'],c=colors[0],clip=(0,1),linewidth=lw,linestyle='-.')
sns.kdeplot(data=chip_3['q_chip'],c=colors[1],clip=(0,1),linewidth=lw,linestyle='-.')
sns.kdeplot(data=chip_3['q_chip_uncor'],c=colors[4],clip=(0,1),linewidth=lw,linestyle='-.')
sns.kdeplot(data=chip_3['q_single'],c=colors[2],clip=(0,1),linewidth=lw,linestyle='-.')
sns.kdeplot(data=chip_3['prob_neg_drop_one_det'],c=colors[3],clip=(0,1),linewidth=lw,linestyle='-.')



plt.xlabel('q')
plt.ylabel('pdf')


plt.rcParams.update({'font.size': 30})
plt.legend(frameon=False,bbox_to_anchor=(0.15,0.2))

plt.xlim([-0.005,1.005])
plt.ylim([0,30])

In [ ]:
#warning this will overwrite the data provided

chip_1.to_csv(f'NegBinomValidation/chip_{q_1}.csv')
chip_2.to_csv(f'NegBinomValidation/chip_{q_2}.csv')
chip_3.to_csv(f'NegBinomValidation/chip_{q_3}.csv')

with open(f'NegBinomValidation/Cov_{q_1}.npy', 'wb') as f:
        np.save(f,Cov_1) 

with open(f'NegBinomValidation/Cov_{q_2}.npy', 'wb') as f:
        np.save(f,Cov_2) 

with open(f'NegBinomValidation/Cov_{q_3}.npy', 'wb') as f:
        np.save(f,Cov_3) 
            

# Distance to Sampling distribution with Jensen–Shannon divergence

In [ ]:
from scipy.stats import differential_entropy

def KS_sample_to_gaussian(sample,mu,sigma):
    # KS divergence from sample to Gaussian with mean mu and std sigma
    # requires entropy estimation of sample
    entropy=differential_entropy(sample,method='ebrahimi')
    window_length=int(np.floor(np.sqrt(len(sample)) + 0.5)) # default window length
    max_length=len(sample)//2-1
    while(np.isinf(entropy) and  (window_length < max_length)):
        window_length=window_length+1
        entropy=differential_entropy(sample,method='ebrahimi',window_length=window_length)
    if(window_length==max_length): # terminated because of max but still infiinte
        return np.inf
    foo=(np.var(sample) + (np.mean(sample) - mu)**2 )/sigma**2
    foo2=0.5 * (np.log(2 * np.pi * sigma**2) + foo )
    return (foo2 - entropy)

In [ ]:
N_droplets_per_chip=500
Num_chips=200
lambda_NegBinom=3
#lambda_Poiss=3

r=2

In [ ]:
num_samples=10000 # for Monte_Carlo estimate of Fisher Informations


q = np.arange(0.05,1, 0.05)
detection_prob=np.arange(0.3,1, 0.05)


In [ ]:

# Poisson
for i in range(len(detection_prob)): # that is detection_prob
    KS_q=np.zeros_like(q)
    KS_rho=np.zeros_like(q)    

    KS_q_c=np.zeros_like(q)
    KS_rho_c=np.zeros_like(q)   
    
    for j in range(len(q)): # q 
        print(i,j)       
        chip = pd.concat(map(lambda x: CreateAndAnalyseChip_Poiss(lambda_Poiss,q[j],detection_prob[i],N_droplets_per_chip), np.arange(Num_chips)),ignore_index=True)
        Fisher_Information=Fisher_Information_Poiss([lambda_Poiss * (1-detection_prob[i]),q[j]],lambda_Poiss *  detection_prob[i],num_samples) 

        Cov=ComputeCov_Poiss(Fisher_Information,N_droplets_per_chip,lambda_Poiss * detection_prob[i],detection_prob[i],lambda_Poiss,q[j]) # includes sample size 
        if(Cov[2,2]):
            
            KS_q[j]=KS_sample_to_gaussian(chip['q_chip_like'].values,q[j],np.sqrt(Cov[2,2]))
            KS_q_c[j]=KS_sample_to_gaussian(chip['q_chip'].values,q[j],np.sqrt(Cov[2,2]))

            KS_q_c_un[j]=KS_sample_to_gaussian(chip['q_chip_uncor'].values,q[j],np.sqrt(Cov[2,2]))
            KS_q_s[j]=KS_sample_to_gaussian(chip['q_single'].values,q[j],np.sqrt(Cov[2,2]))
            KS_q_s_un[j]=KS_sample_to_gaussian(chip['prob_neg_drop_one_det'].values,q[j],np.sqrt(Cov[2,2]))

            
            
            
        else: #zero covariance gives delta 
            KS_q[j]=np.inf
            KS_q_c[j]=np.inf
            KS_q_c_un[j]=np.inf
            KS_q_s[j]=np.inf
            KS_q_s_un[j]=np.inf


        if(Cov[1,1]):
            KS_rho[j]=KS_sample_to_gaussian(chip['rho_like'].values,detection_prob[i],np.sqrt(Cov[1,1]))
            KS_rho_c[j]=KS_sample_to_gaussian(chip['rho'].values,detection_prob[i],np.sqrt(Cov[1,1]))
        else:
            KS_rho[j]=np.inf
            KS_rho_c[j]=np.inf
            
  
    with open(f'KS_q_{detection_prob[i]}.npy', 'wb') as f:
        np.save(f,KS_q) 
        
    with open(f'KS_q_c_{detection_prob[i]}.npy', 'wb') as f:
        np.save(f,KS_q_c) 

    with open(f'KS_q_c_un_{detection_prob[i]}.npy', 'wb') as f:
        np.save(f,KS_q_c_un) 
        
    with open(f'KS_q_s_{detection_prob[i]}.npy', 'wb') as f:
        np.save(f,KS_q_s) 

    with open(f'KS_q_s_un_{detection_prob[i]}.npy', 'wb') as f:
        np.save(f,KS_q_s_un) 
        
    with open(f'KS_rho_{detection_prob[i]}.npy', 'wb') as f:
        np.save(f,KS_rho) 
        
    with open(f'KS_rho_c_{detection_prob[i]}.npy', 'wb') as f:
        np.save(f,KS_rho_c) 
        
    print(f'save {i},{detection_prob[i]} done')
'

In [ ]:
#warning this will overwrite provided data

for i in range(len(detection_prob)): # that is detection_prob
    KS_q=np.zeros_like(q)
    KS_rho=np.zeros_like(q)    
    FisherInfo_r=  N_droplets_per_chip * Estimate_Fisher_Information_part_r(lambda_NegBinom * detection_prob[i],r,num_samples) 

    KS_q_c=np.zeros_like(q)
    KS_rho_c=np.zeros_like(q)   

    KS_q_c_un=np.zeros_like(q)
    KS_q_s_un=np.zeros_like(q)
    KS_q_s=np.zeros_like(q)
    
    for j in range(len(q)): # q
        print(i,j)
        chip = pd.concat(map(lambda x: CreateAndAnalyseChip_NegBinom(lambda_NegBinom,r,q[j],detection_prob[i],N_droplets_per_chip), np.arange(Num_chips)),ignore_index=True)
        Fisher_Information=   Fisher_Information_NegBinom([(lambda_NegBinom * (1-detection_prob[i]))/(r+lambda_NegBinom * detection_prob[i]),q[j]],lambda_NegBinom * detection_prob[i],r,num_samples)
        Cov=ComputeCov_NegBinom(Fisher_Information,FisherInfo_r,N_droplets_per_chip,lambda_NegBinom * detection_prob[i],detection_prob[i],lambda_NegBinom,r,(lambda_NegBinom * (1-detection_prob[i]))/(r+lambda_NegBinom * detection_prob[i]),q[j])
        if(Cov[3,3]):
            KS_q[j]=KS_sample_to_gaussian(chip['q_chip_like'].values,q[j],np.sqrt(Cov[3,3]))
            KS_q_c[j]=KS_sample_to_gaussian(chip['q_chip'].values,q[j],np.sqrt(Cov[3,3]))
            
            KS_q_c_un[j]=KS_sample_to_gaussian(chip['q_chip_uncor'].values,q[j],np.sqrt(Cov[3,3]))
            KS_q_s[j]=KS_sample_to_gaussian(chip['q_single'].values,q[j],np.sqrt(Cov[3,3]))
            KS_q_s_un[j]=KS_sample_to_gaussian(chip['prob_neg_drop_one_det'].values,q[j],np.sqrt(Cov[3,3]))

        
        else: #zero covariance gives delta 
            KS_q[j]=np.inf
            KS_q_c[j]=np.inf
            KS_q_c_un[j]=np.inf
            KS_q_s[j]=np.inf
            KS_q_s_un[j]=np.inf

        if(Cov[2,2]):
            KS_rho[j]=KS_sample_to_gaussian(chip['rho_like'].values,detection_prob[i],np.sqrt(Cov[2,2]))
            KS_rho_c[j]=KS_sample_to_gaussian(chip['rho'].values,detection_prob[i],np.sqrt(Cov[2,2]))
        else:
            KS_rho[j]=np.inf
            KS_rho_c[j]=np.inf
            

    with open(f'NegBinomValidation/KS_q_{detection_prob[i]}.npy', 'wb') as f:
        np.save(f,KS_q) 
        
    with open(f'NegBinomValidation/KS_q_c_{detection_prob[i]}.npy', 'wb') as f:
        np.save(f,KS_q_c) 

    with open(f'NegBinomValidation/KS_q_c_un_{detection_prob[i]}.npy', 'wb') as f:
        np.save(f,KS_q_c_un) 
        
    with open(f'NegBinomValidation/KS_q_s_{detection_prob[i]}.npy', 'wb') as f:
        np.save(f,KS_q_s) 

    with open(f'NegBinomValidation/KS_q_s_un_{detection_prob[i]}.npy', 'wb') as f:
        np.save(f,KS_q_s_un) 
        
    with open(f'NegBinomValidation/KS_rho_{detection_prob[i]}.npy', 'wb') as f:
        np.save(f,KS_rho) 
        
    with open(f'NegBinomValidation/KS_rho_c_{detection_prob[i]}.npy', 'wb') as f:
        np.save(f,KS_rho_c) 
        
    print(f'save {i},{detection_prob[i]} done')